# Task 2: Data splitting (KIBA)

Input: `data/processed/kiba_pairs_clean.csv`, the Task 1 output (117,657 pairs, 2,068 ligands after merging stereoisomers, 229 targets). The split is `70 | 15 | 15` for `train | val | test`.

Rules from the brief:
1. No ligand–target pair may occur in more than one partition.
2. Every evaluation ligand and target must occur in training.

The split is made once here and saved. The CF notebook only loads it.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42
ROOT = next(p for p in [Path(".."), Path(".")] if (p / "data" / "processed" / "kiba_pairs_clean.csv").exists())
out_dir = ROOT / "data" / "splits"
out_dir.mkdir(parents=True, exist_ok=True)

data = pd.read_csv(ROOT / "data" / "processed" / "kiba_pairs_clean.csv")
ligands = pd.read_csv(ROOT / "data" / "processed" / "kiba_ligands.csv")
targets = pd.read_csv(ROOT / "data" / "processed" / "kiba_targets.csv")
print(data.shape, "| ligands:", data.ligand_id.nunique(), "| targets:", data.target_id.nunique())

(117657, 11) | ligands: 2068 | targets: 229


`lig_id` / `tgt_id` are the row / column indices of the CF matrices. They follow the order of `kiba_ligands.csv` and `kiba_targets.csv`, so they don't depend on row order in the pairs file. `ligand_id` (CHEMBL) and `target_id` (UniProt) are kept next to them.

**Stratification.** On DAVIS the split is stratified by floor vs measured pairs. KIBA has no hard floor, but Task 1 found that 22.9% of the scores sit exactly at 11.1 or 11.2 (the heaped "spike" values). We stratify by that flag, so every partition has the same share of spike pairs.

In [2]:
data["lig_id"] = pd.Categorical(data.ligand_id, categories=ligands.ligand_id).codes
data["tgt_id"] = pd.Categorical(data.target_id, categories=targets.target_id).codes
assert (data.lig_id >= 0).all() and (data.tgt_id >= 0).all(), "every pair must map to a known ligand and target"
assert not data.duplicated(["lig_id", "tgt_id"]).any()

data["pair_type"] = np.where(data.flag_spike, "spike", "other")

# cut 1: 70% train / 30% rest
# cut 2: split the rest in half -> 15% val / 15% test
train_idx, rest_idx = train_test_split(data.index, test_size=0.30, random_state=SEED,
                                       stratify=data["pair_type"])
val_idx, test_idx = train_test_split(rest_idx, test_size=0.50, random_state=SEED,
                                     stratify=data.loc[rest_idx, "pair_type"])
data["split"] = "train"
data.loc[val_idx, "split"] = "val"
data.loc[test_idx, "split"] = "test"

**Coverage rule.** KIBA is sparse: the sparsest ligand has only 11 measured targets. A random split can leave a ligand with no training pair at all. Any val/test pair whose ligand or target has no training pair is moved into train. This only ever adds to train, so one pass per column is enough.

In [3]:
moved = {}
for col in ["lig_id", "tgt_id"]:
    in_train = set(data.loc[data.split == "train", col])
    bad = (data.split != "train") & ~data[col].isin(in_train)
    moved[col] = int(bad.sum())
    data.loc[bad, "split"] = "train"
print("pairs moved into train to satisfy the coverage rule:", moved)

pairs moved into train to satisfy the coverage rule: {'lig_id': 0, 'tgt_id': 0}


## Verify

In [4]:
parts = {n: data[data.split == n] for n in ["train", "val", "test"]}
pairs = {n: set(zip(p.lig_id, p.tgt_id)) for n, p in parts.items()}

# 1) no ligand-target pair in more than one partition
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    assert not pairs[a] & pairs[b], f"pairs shared by {a} and {b}"
assert sum(len(p) for p in pairs.values()) == len(data)

# 2) every ligand and target in val/test also occurs in train
for name in ["val", "test"]:
    assert parts[name].lig_id.isin(parts["train"].lig_id).all()
    assert parts[name].tgt_id.isin(parts["train"].tgt_id).all()
print("checks passed: no pair in two partitions; every val/test ligand and target appears in train")

checks passed: no pair in two partitions; every val/test ligand and target appears in train


**Split counts table** (for the report).

In [5]:
counts = pd.DataFrame({p: {"pairs": len(d), "share of pairs": f"{len(d) / len(data):.1%}",
                           "ligands": d.lig_id.nunique(), "targets": d.tgt_id.nunique(),
                           "share spike (11.1 / 11.2)": f"{(d.pair_type == 'spike').mean():.1%}",
                           "mean score": round(d.score.mean(), 3)}
                       for p, d in parts.items()}).T
counts.loc["total"] = [len(data), "100.0%", data.lig_id.nunique(), data.tgt_id.nunique(),
                       f"{(data.pair_type == 'spike').mean():.1%}", round(data.score.mean(), 3)]
counts["pairs moved for coverage"] = ["", "", "", sum(moved.values())]
counts

,pairs,share of pairs,ligands,targets,share spike (11.1 / 11.2),mean score,pairs moved for coverage
train,82359,70.0%,2068,229,22.9%,11.722,
val,17649,15.0%,1971,228,22.9%,11.719,
test,17649,15.0%,1953,226,22.9%,11.721,
total,117657,100.0%,2068,229,22.9%,11.721,0


Unlike DAVIS, not every ligand appears in val and test: a ligand with 11 pairs gets only one or two evaluation pairs, or none. Training pairs per ligand are very uneven, which is what the sparsity analysis (E3) in the CF notebook bins on.

In [6]:
train_per_lig = parts["train"].groupby("lig_id").size()
print("training pairs per ligand:", train_per_lig.describe().round(1).to_dict())
print("ligands with test pairs:", parts["test"].lig_id.nunique(), "of", data.lig_id.nunique())

training pairs per ligand: {'count': 2068.0, 'mean': 39.8, 'std': 34.4, 'min': 4.0, '25%': 10.0, '50%': 22.0, '75%': 71.2, 'max': 123.0}
ligands with test pairs: 1953 of 2068


## Save

In [7]:
cols = ["lig_id", "tgt_id", "ligand_id", "target_id", "score", "pair_type", "flag_far_low", "flag_zero",
        "flag_far_high", "flag_low_ligand", "n_merged", "split"]
data[cols].sort_values(["lig_id", "tgt_id"]).to_csv(out_dir / "kiba_split.csv", index=False)
counts.to_csv(ROOT / "results" / "kiba" / "task2_split_counts.csv")
print("saved", data[cols].shape, "to", out_dir / "kiba_split.csv")

saved (117657, 12) to ..\data\splits\kiba_split.csv
